# Script to show plots for world map

In [1]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np

In [2]:
import plotly.graph_objects as go
import plotly.io as pio

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import requests
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.lines import Line2D

from bar_map_functions import *

### Parameters to change according to final plot 

In [3]:
save_output = False
focus = "consumption" # Change to "production" or "consumption"
scenarios = ["2021", "2024", "2035 High Demand"]

In [4]:
title_plotly = f"world_map_plotly_{focus}.png"
title_mpl = f"world_map_mpl_{focus}.png"

### Load all relevant files

In [5]:
output_path = "../../02_plots"

In [6]:
input_file_path = os.path.join('..', '..', '01_data', '01_input_data', '02_processed', '01_paper_IAEE', '01_data_sheets_input')
input_file = f"\\paper_paris_2025_input_{focus}_world.xlsx"
full_input_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path + input_file))

data_gas_prod = pd.read_excel(full_input_path)

In [7]:
LNG_location_path = os.path.join('..', '..','01_data', '01_input_data', '01_raw', '01_Russian_War_Case')
LNG_file = '\\LNG_locations.xlsx'
full_LNG_path = os.path.abspath(os.path.join(LNG_location_path + LNG_file))

ports_df = pd.read_excel(full_LNG_path, sheet_name="Global")

In [8]:
shapefile_path = os.path.join('..', '..','01_data', '01_input_data', '01_raw', 'world_countries_shapefile')
shapefile = '\\ne_50m_admin_0_countries_lakes.shp'
full_shapefile_path = os.path.abspath(os.path.join(os.getcwd(), shapefile_path + shapefile))

world = gpd.read_file(full_shapefile_path)

### Fix files for better visualization

In [9]:
ukraine_json_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_UKR_0.json"
ukraine_json_path = 'gadm41_UKR_0.json'
response = requests.get(ukraine_json_url)
with open(ukraine_json_path, "wb") as f:
    f.write(response.content)
ukraine_corrected = gpd.read_file(ukraine_json_path)

In [10]:
fix_iso = {"France": "FRA", "Norway": "NOR", "Kosovo": "XKX"}

In [11]:
world = clean_world_file(world, ukraine_corrected, fix_iso)

In [12]:
ports_df['Latitude'] = ports_df['Latitude'].astype(str).str.replace(',', '.').astype(float)
ports_df['Longitude'] = ports_df['Longitude'].astype(str).str.replace(',', '.').astype(float)

In [13]:
region_to_countries = {
    "Africa": world[world["CONTINENT"] == "Africa"]["ISO_A3"].tolist(),
    "North America": world[world["CONTINENT"] == "North America"]["ISO_A3"].tolist(),
    "South America": world[world["CONTINENT"] == "South America"]["ISO_A3"].tolist(),
    "Asia": [c for c in world[world["CONTINENT"] == "Asia"]["ISO_A3"].tolist() if c not in ["TUR"]] + ["PNG"],
    "Australia": ["AUS", "NZL"],
    "Caspian Region": ["AZE", "KAZ", "UZB", "KGZ", "TKM", "TJK", "GEO"],
    "Middle East": ["SAU", "QAT", "UAE", "KWT", "OMN", "IRQ", "ISR", "JOR", "SYR", "LBN", "YEM", "ARE", "IRN"],
    "Russia": ["RUS"],
}

### Define all colors and heights

In [14]:
pastel_colors = {
    "Africa": "#c7e9c0",
    "North America": "#bae4f5",
    "South America": "#7cc2f3",
    "Middle East": "#fdd0a2",
    "Caspian Region": "#fdae6b",
    "Asia": "#fcbba1",
    "Russia": "#fee6ce",
    "Australia": "#dadaeb",
}

In [15]:
highlight_countries = {
    "USA": "#2e8ccf", "TTO": "#2e3bcf", "QAT": "#e6550d",
    "DZA": "#247143", "EGY": "#247143", "MAR": "#247143", "LBY": "#247143", "TUN": "#247143",
    "MYS": "#e36c3d", "IND": "#e36c3d", "CHN": "#e36c3d", "HKG": "#e36c3d", "IDN": "#e36c3d",
    "KOR": "#dd7762", "JPN": "#dd7762",
}

In [16]:
config = map_config(data_gas_prod, scenarios)
config["bar_height_scale"] = 28
config["scale_lon"] = -150
config["bar_spacing"] = 3

#### Store the colors to regions and countries

In [17]:
iso_to_color = {}
for region, isos in region_to_countries.items():
    color = pastel_colors.get(region, "#cccccc")
    for iso in isos:
        iso_to_color[iso] = color

In [18]:
for iso in region_to_countries["Africa"]:  # force all Africa to be green
    iso_to_color[iso] = pastel_colors["Africa"]

In [19]:
world["fill_color"] = world["ISO_A3"].map(iso_to_color).fillna("#e0e0e0")

### Build Plotly figure

In [ ]:
build_plotly_map(data_gas_prod, world, ports_df, region_to_countries, highlight_countries, scenarios, config, output_path, title_plotly, save_output)